Nome: Maria Eduarda Rezende Rodrigues
RA: 165873

Importação das bibliotecas usadas no notebook: pandas para manipulação de tabelas, numpy para operações numéricas e vetores, e math para funções matemáticas básicas como cosseno, exponencial e verificação de valores finitos.

In [ ]:
import pandas as pd
import numpy as np
import math

Definição do método da bisseção. A função recebe f, o intervalo [a, b], a tolerância e o número máximo de iterações. Primeiro são feitas as verificações que impedem a execução: a maior ou igual a b, tolerância ou número de iterações inválido, função não finita nas extremidades, e ausência de mudança de sinal quando f(a)*f(b) é maior que zero. Também trata os casos em que a ou b já é raiz. A cada iteração calcula o ponto médio m, avalia f(m) e reduz o intervalo pela metade escolhendo o lado onde ocorre a troca de sinal. A parada acontece quando f(m) é zero ou quando a largura do intervalo dividida por dois fica menor ou igual à tolerância. Retorna a raiz, o número de iterações, a lista de aproximações e o status.

In [11]:
def bissecao(f, a, b, tol=1e-6, max_iter=100):
    """Resolve f(x)=0 pelo método da bisseção.

    Retorna (raiz, iteracoes, aproximacoes, status).
    """
    if a >= b:
        return None, 0, [], "falha: é necessário fornecer a < b"

    if tol <= 0 or max_iter < 1:
        return None, 0, [], "falha: tolerância ou número de iterações inválido"

    fa, fb = f(a), f(b)

    if not _finito(fa, fb):
        return None, 0, [], "falha: função não finita nas extremidades"

    if fa == 0:
        return a, 0, [a], "convergiu: a é raiz"

    if fb == 0:
        return b, 0, [b], "convergiu: b é raiz"

    if fa * fb > 0:
        return None, 0, [], "falha: não há mudança de sinal"

    aproximacoes = []

    for k in range(1, max_iter + 1):
        m = a + (b - a) / 2
        fm = f(m)
        aproximacoes.append(m)

        erro = (b - a) / 2  # critério do passo (largura do intervalo)

        if fm == 0 or erro <= tol:
            return m, k, aproximacoes, "convergiu"

        if fa * fm < 0:
            b, fb = m, fm
        else:
            a, fa = m, fm

    return m, max_iter, aproximacoes, "falha: máximo de iterações atingido"

Definição do método de Newton. A função recebe f, sua derivada df, o chute inicial x0, a tolerância, o número máximo de iterações e uma tolerância para a derivada. Antes do laço verifica se o chute inicial já satisfaz o critério do resíduo. A cada iteração calcula a derivada em x e, se ela for nula ou muito pequena, interrompe com falha porque não é possível continuar. Caso contrário, gera a nova aproximação usando a fórmula de Newton, guarda o valor e verifica dois critérios de parada: resíduo pequeno, que indica convergência, ou passo pequeno com resíduo ainda alto, que indica estagnação longe da raiz. Se nenhum critério for atingido dentro do limite de iterações, retorna falha por máximo de iterações.

In [12]:
def newton(f, df, x0, tol=1e-6, max_iter=100, deriv_tol=1e-14):
    """Resolve f(x)=0 pelo método de Newton.

    Retorna (raiz, iteracoes, aproximacoes, status).
    """
    if tol <= 0 or max_iter < 0:
        return None, 0, [], "falha: tolerância ou número de iterações inválido"

    x = float(x0)
    aproximacoes = []

    fx = f(x)
    if abs(fx) <= tol:  # critério do resíduo já satisfeito no chute inicial
        return x, 0, aproximacoes, "convergiu pelo resíduo"

    for k in range(1, max_iter + 1):
        dfx = df(x)

        if abs(dfx) <= deriv_tol:  # derivada nula/pequena -> não dá para continuar
            return x, k - 1, aproximacoes, "falha: derivada nula ou muito pequena"

        x_novo = x - fx / dfx
        aproximacoes.append(x_novo)
        f_novo = f(x_novo)

        if abs(f_novo) <= tol:  # critério do resíduo
            return x_novo, k, aproximacoes, "convergiu pelo resíduo"

        if abs(x_novo - x) <= tol * max(1.0, abs(x_novo)):  # passo pequeno
            return x_novo, k, aproximacoes, "falha: passo pequeno, mas resíduo ainda alto"

        x, fx = x_novo, f_novo

    return x, max_iter, aproximacoes, "falha: máximo de iterações atingido"

Definição do método das secantes. A função recebe f, dois chutes iniciais x0 e x1, a tolerância, o número máximo de iterações e uma tolerância para o denominador. A cada iteração avalia f nos dois pontos e monta o denominador como a diferença entre esses valores. Se o denominador for nulo ou muito pequeno, interrompe com falha, pois a reta secante fica indefinida. Caso contrário, calcula a nova aproximação pela fórmula da secante e verifica os mesmos dois critérios de parada usados no Newton: resíduo pequeno indicando convergência e passo pequeno com resíduo alto indicando estagnação. Ao final de cada passo, desloca os dois pontos para a próxima iteração. Se não convergir, retorna falha por máximo de iterações.

In [13]:
def secantes(f, x0, x1, tol=1e-6, max_iter=100, denom_tol=1e-14):
    """Resolve f(x)=0 pelo método das secantes.

    Retorna (raiz, iteracoes, aproximacoes, status).
    """
    if tol <= 0 or max_iter < 1:
        return None, 0, [], "falha: tolerância ou número de iterações inválido"

    x_anterior = float(x0)
    x_atual = float(x1)
    aproximacoes = []

    for k in range(1, max_iter + 1):
        f_anterior = f(x_anterior)
        f_atual = f(x_atual)

        if abs(f_atual) <= tol:  # critério do resíduo
            return x_atual, k - 1, aproximacoes, "convergiu pelo resíduo"

        denominador = f_atual - f_anterior

        if abs(denominador) <= denom_tol:  # denominador nulo/pequeno
            return x_atual, k - 1, aproximacoes, \
                   "falha: denominador nulo ou muito pequeno"

        x_novo = x_atual - f_atual * (x_atual - x_anterior) / denominador
        aproximacoes.append(x_novo)

        f_novo = f(x_novo)

        if abs(f_novo) <= tol:  # critério do resíduo
            return x_novo, k, aproximacoes, "convergiu pelo resíduo"

        if abs(x_novo - x_atual) <= tol * max(1.0, abs(x_novo)):  # passo pequeno
            return x_novo, k, aproximacoes, \
                   "falha: passo pequeno, mas resíduo ainda alto"

        x_anterior, x_atual = x_atual, x_novo

    return x_atual, max_iter, aproximacoes, \
           "falha: máximo de iterações atingido"

Critérios de parada adotados

Foram combinados dois critérios de parada para aceitar uma raiz:

1. Resíduo (valor da função): aceitamos $x_{k+1}$ como raiz quando

$$|f(x_{k+1})| \le \text{tol}.$$

   Este é o critério principal, pois mede diretamente o quão perto de zero a função está — que é exatamente o que se deseja resolver.

Montagem dos casos de teste. Cada item da lista descreve uma função, sua derivada e os valores iniciais adequados para cada método: o intervalo da bisseção, o chute único do Newton e os dois chutes das secantes. São usadas quatro funções diferentes: x^2 - 2, cos(x) - x, x^3 - x - 2 e e^x - 3x. Essa estrutura permite aplicar os três métodos de forma automática a todas as funções na etapa seguinte.

In [14]:
# Cada caso define a função, sua derivada e os valores iniciais de cada método.
casos = [
    {
        "nome": "x^2 - 2",
        "f": lambda x: x**2 - 2,
        "df": lambda x: 2*x,
        "bissecao": (0.0, 2.0),
        "newton": (1.0,),
        "secantes": (1.0, 2.0),
    },
    {
        "nome": "cos(x) - x",
        "f": lambda x: math.cos(x) - x,
        "df": lambda x: -math.sin(x) - 1,
        "bissecao": (0.0, 1.0),
        "newton": (0.5,),
        "secantes": (0.0, 1.0),
    },
    {
        "nome": "x^3 - x - 2",
        "f": lambda x: x**3 - x - 2,
        "df": lambda x: 3*x**2 - 1,
        "bissecao": (1.0, 2.0),
        "newton": (1.5,),
        "secantes": (1.0, 2.0),
    },
    {
        "nome": "e^x - 3x",
        "f": lambda x: math.exp(x) - 3*x,
        "df": lambda x: math.exp(x) - 3,
        "bissecao": (0.0, 1.0),
        "newton": (0.5,),
        "secantes": (0.0, 1.0),
    },
]

Definição das funções auxiliares e coleta dos resultados. A função _finito verifica se todos os valores passados são finitos, ou seja, não são infinito nem indefinido, e é usada pela bisseção. A função coletar_resultados percorre todos os casos e aplica os três métodos a cada um, montando uma lista de linhas com o nome da função, o método, os valores iniciais, a raiz encontrada, o número de iterações e o status. A função resid recalcula o resíduo |f(x)| final para a raiz obtida, localizando a função correspondente pelo nome do caso. Ao final, a chamada de coletar_resultados executa tudo e guarda as linhas.

In [17]:
import math

def _finito(*valores):
    """Retorna True se todos os valores forem finitos (nem inf, nem nan)."""
    return all(math.isfinite(v) for v in valores)

def coletar_resultados(casos):
    """Aplica os três métodos a cada caso e devolve uma lista de linhas da tabela."""
    linhas = []
    for caso in casos:
        f, df = caso["f"], caso["df"]

        # Bisseção
        a, b = caso["bissecao"]
        raiz, iters, _, status = bissecao(f, a, b)
        linhas.append((caso["nome"], "Bisseção", f"[{a}, {b}]", raiz, iters, status))

        # Newton
        (x0,) = caso["newton"]
        raiz, iters, _, status = newton(f, df, x0)
        linhas.append((caso["nome"], "Newton", f"x0={x0}", raiz, iters, status))

        # Secantes
        x0, x1 = caso["secantes"]
        raiz, iters, _, status = secantes(f, x0, x1)
        linhas.append((caso["nome"], "Secantes", f"x0={x0}, x1={x1}", raiz, iters, status))
    return linhas


def resid(caso_nome, raiz):
    """Calcula |f(x)| final para a raiz encontrada, buscando f pelo nome do caso."""
    if raiz is None:
        return None
    f = next(c["f"] for c in casos if c["nome"] == caso_nome)
    return abs(f(raiz))


linhas = coletar_resultados(casos)

Impressão da tabela de resultados. São definidos os títulos das colunas e a largura de cada uma. A função fmt ajusta cada valor ao tamanho da coluna. O código imprime uma linha separadora, o cabeçalho e outra separadora, e em seguida percorre todas as linhas coletadas: para cada uma calcula o resíduo final, formata a raiz e o resíduo, monta o texto de iterações com o status e imprime tudo alinhado. As colunas apresentadas são função, método, valores iniciais, raiz aproximada, |f(x)| final e iterações/resultado.

In [18]:
# Impressão da tabela em texto puro (sem dependências externas).
cab = ("Função", "Método", "Valores iniciais", "Raiz aproximada", "|f(x)| final", "Iterações/resultado")
larg = (12, 9, 18, 18, 14, 40)

def fmt(valor, w):
    return str(valor).ljust(w)[:w] if w else str(valor)

linha_sep = "-" * (sum(larg) + len(larg))
print(linha_sep)
print(" ".join(fmt(c, w) for c, w in zip(cab, larg)))
print(linha_sep)
for nome, metodo, iniciais, raiz, iters, status in linhas:
    r = resid(nome, raiz)
    raiz_txt = f"{raiz:.10f}" if raiz is not None else "—"
    resid_txt = f"{r:.2e}" if r is not None else "—"
    iters_txt = f"{iters} it. | {status}"
    print(" ".join(fmt(v, w) for v, w in zip(
        (nome, metodo, iniciais, raiz_txt, resid_txt, iters_txt), larg)))
print(linha_sep)

---------------------------------------------------------------------------------------------------------------------
Função       Método    Valores iniciais   Raiz aproximada    |f(x)| final   Iterações/resultado                     
---------------------------------------------------------------------------------------------------------------------
x^2 - 2      Bisseção  [0.0, 2.0]         1.4142141342       1.62e-06       21 it. | convergiu                      
x^2 - 2      Newton    x0=1.0             1.4142135624       4.51e-12       4 it. | convergiu pelo resíduo          
x^2 - 2      Secantes  x0=1.0, x1=2.0     1.4142135621       8.93e-10       5 it. | convergiu pelo resíduo          
cos(x) - x   Bisseção  [0.0, 1.0]         0.7390851974       1.08e-07       20 it. | convergiu                      
cos(x) - x   Newton    x0=0.5             0.7390851339       1.18e-09       3 it. | convergiu pelo resíduo          
cos(x) - x   Secantes  x0=0.0, x1=1.0     0.7390851121       3

Versões dos métodos preparadas para animação. Estas funções repetem a lógica dos métodos originais, mas em vez de retornar apenas a raiz, elas guardam o histórico geométrico de cada iteração, que será usado para desenhar os gráficos. A bissecao_traco registra o intervalo, o ponto médio e o valor da função em cada passo. A newton_traco registra o ponto atual, o valor da função, a derivada e o próximo ponto, dados necessários para desenhar a reta tangente. A secantes_traco registra os dois pontos usados e o novo ponto, permitindo desenhar a reta secante. As funções originais das etapas anteriores não são alteradas.

In [19]:
def bissecao_traco(f, a, b, tol=1e-6, max_iter=100):
    """Igual à bisseção, mas registra [a, b], ponto médio e f(m) por iteração."""
    fa, fb = f(a), f(b)
    if fa * fb > 0:
        return []
    passos = []
    for k in range(1, max_iter + 1):
        m = a + (b - a) / 2
        fm = f(m)
        passos.append({"iter": k, "a": a, "b": b, "x": m, "fx": fm})
        if fm == 0 or (b - a) / 2 <= tol:
            break
        if fa * fm < 0:
            b, fb = m, fm
        else:
            a, fa = m, fm
    return passos


def newton_traco(f, df, x0, tol=1e-6, max_iter=100, deriv_tol=1e-14):
    """Registra x_k, f(x_k), f'(x_k) e o próximo x para desenhar a tangente."""
    x = float(x0)
    passos = []
    for k in range(1, max_iter + 1):
        fx = f(x)
        dfx = df(x)
        if abs(dfx) <= deriv_tol:
            break
        x_novo = x - fx / dfx
        passos.append({"iter": k, "x": x, "fx": fx, "dfx": dfx, "x_novo": x_novo})
        if abs(f(x_novo)) <= tol or abs(x_novo - x) <= tol * max(1.0, abs(x_novo)):
            break
        x = x_novo
    return passos


def secantes_traco(f, x0, x1, tol=1e-6, max_iter=100, denom_tol=1e-14):
    """Registra os dois pontos (x0, x1) e o novo x para desenhar a secante."""
    xa, xb = float(x0), float(x1)
    passos = []
    for k in range(1, max_iter + 1):
        fa, fb = f(xa), f(xb)
        denom = fb - fa
        if abs(denom) <= denom_tol:
            break
        x_novo = xb - fb * (xb - xa) / denom
        passos.append({"iter": k, "x0": xa, "x1": xb, "fx0": fa, "fx1": fb, "x_novo": x_novo})
        if abs(f(x_novo)) <= tol or abs(x_novo - xb) <= tol * max(1.0, abs(x_novo)):
            break
        xa, xb = xb, x_novo
    return passos

Preparação do motor de animação. O código importa os módulos necessários, define o backend Agg do matplotlib para permitir salvar figuras sem tela, e cria a pasta animacoes onde os arquivos GIF serão guardados. A função _base_plot desenha a curva de f(x) e o eixo horizontal y=0. A função _texto_info coloca no título de cada quadro o número da iteração, o valor atual da aproximação e o resíduo. A função animar monta a figura, define os limites dos eixos e cria uma função interna desenhar que, para cada quadro, mostra os elementos geométricos do método escolhido: o intervalo destacado na bisseção, a reta tangente no Newton e a reta secante nas secantes, além dos pontos das aproximações já visitadas. Ao final, a animação é gerada quadro a quadro e salva como GIF usando o PillowWriter.

In [20]:
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")  # backend não-interativo, adequado para salvar GIFs
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

os.makedirs("animacoes", exist_ok=True)


def _base_plot(ax, f, xlim):
    """Desenha f(x) e o eixo y=0 nos limites dados."""
    xs = np.linspace(xlim[0], xlim[1], 400)
    ys = [f(x) for x in xs]
    ax.plot(xs, ys, color="#1f77b4", lw=2, label="f(x)")
    ax.axhline(0, color="black", lw=1)  # eixo horizontal y = 0
    ax.grid(alpha=0.3)
    ax.set_xlim(*xlim)
    return ys


def _texto_info(ax, passo, valor, residuo):
    ax.set_title(
        f"Iteração {passo['iter']}  |  x = {valor:.8f}  |  |f(x)| = {residuo:.2e}"
    )


def animar(metodo, f, passos, xlim, titulo, arquivo, df=None, fps=1.2):
    """Gera e salva um GIF da evolução do método.

    metodo: 'bissecao', 'newton' ou 'secantes'.
    """
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ys = _base_plot(ax, f, xlim)
    ymin, ymax = min(ys), max(ys)
    margem = 0.15 * (ymax - ymin + 1e-9)
    ax.set_ylim(ymin - margem, ymax + margem)
    ax.set_xlabel("x"); ax.set_ylabel("f(x)")
    fig.suptitle(titulo, fontweight="bold")

    aprox_x, aprox_y = [], []  # pontos já visitados

    def desenhar(i):
        # remove artistas dinâmicos do quadro anterior (mantém f(x), eixo e pontos)
        for art in list(ax.lines) + list(ax.collections):
            if art.get_gid() == "dinamico":
                art.remove()
        p = passos[i]

        if metodo == "bissecao":
            a, b, m = p["a"], p["b"], p["x"]
            # intervalo [a, b] destacado sobre o eixo
            ax.plot([a, b], [0, 0], color="orange", lw=6, alpha=0.5, gid="dinamico")
            ax.axvline(a, color="orange", ls=":", gid="dinamico")
            ax.axvline(b, color="orange", ls=":", gid="dinamico")
            ax.axvline(m, color="red", ls="--", gid="dinamico")
            valor, residuo = m, abs(p["fx"])
            aprox_x.append(m); aprox_y.append(p["fx"])

        elif metodo == "newton":
            x, fx, dfx = p["x"], p["fx"], p["dfx"]
            # reta tangente: y = fx + dfx*(X - x)
            X = np.array(xlim)
            ax.plot(X, fx + dfx * (X - x), color="green", lw=1.5, gid="dinamico",
                    label="tangente")
            ax.axvline(p["x_novo"], color="red", ls="--", gid="dinamico")
            valor, residuo = x, abs(fx)
            aprox_x.append(x); aprox_y.append(fx)

        else:  # secantes
            x0, x1, f0, f1 = p["x0"], p["x1"], p["fx0"], p["fx1"]
            # reta secante pelos dois pontos (x0, f0) e (x1, f1)
            X = np.array(xlim)
            coef = (f1 - f0) / (x1 - x0)
            ax.plot(X, f0 + coef * (X - x0), color="purple", lw=1.5, gid="dinamico",
                    label="secante")
            ax.scatter([x0, x1], [f0, f1], color="purple", zorder=5, gid="dinamico")
            ax.axvline(p["x_novo"], color="red", ls="--", gid="dinamico")
            valor, residuo = x1, abs(f1)
            aprox_x.append(x1); aprox_y.append(f1)

        # pontos das aproximações acumuladas
        ax.scatter(aprox_x, aprox_y, color="red", zorder=6, gid="dinamico")
        _texto_info(ax, p, valor, residuo)
        return ax.lines

    anim = FuncAnimation(fig, desenhar, frames=len(passos), interval=1000 / fps)
    caminho = os.path.join("animacoes", arquivo)
    anim.save(caminho, writer=PillowWriter(fps=fps))
    plt.close(fig)
    print(f"GIF salvo: {caminho}  ({len(passos)} quadros)")
    return caminho

Geração das três animações escolhidas. Cada bloco define a função a ser resolvida, calcula o histórico de passos com a versão de traço correspondente e chama a função animar para produzir o GIF. A primeira animação mostra a bisseção em x^2 - 2 com o intervalo encolhendo. A segunda mostra o método de Newton em x^3 - x - 2 com as retas tangentes. A terceira mostra as secantes em cos(x) - x com as retas secantes. Ao final, os três arquivos GIF ficam salvos na pasta animacoes.

In [21]:
# 1) Bisseção em x^2 - 2
f1 = lambda x: x**2 - 2
passos1 = bissecao_traco(f1, 0.0, 2.0)
animar("bissecao", f1, passos1, xlim=(-0.2, 2.2),
       titulo="Bisseção:  f(x) = x^2 - 2", arquivo="bissecao_x2menos2.gif")

# 2) Newton em x^3 - x - 2
f2 = lambda x: x**3 - x - 2
df2 = lambda x: 3*x**2 - 1
passos2 = newton_traco(f2, df2, 1.5)
animar("newton", f2, passos2, xlim=(0.5, 2.2),
       titulo="Newton:  f(x) = x^3 - x - 2", arquivo="newton_x3menosxmenos2.gif", df=df2)

# 3) Secantes em cos(x) - x
f3 = lambda x: math.cos(x) - x
passos3 = secantes_traco(f3, 0.0, 1.0)
animar("secantes", f3, passos3, xlim=(-0.1, 1.4),
       titulo="Secantes:  f(x) = cos(x) - x", arquivo="secantes_cosxmenozx.gif")

GIF salvo: animacoes/bissecao_x2menos2.gif  (21 quadros)
GIF salvo: animacoes/newton_x3menosxmenos2.gif  (2 quadros)
GIF salvo: animacoes/secantes_cosxmenozx.gif  (4 quadros)


'animacoes/secantes_cosxmenozx.gif'